# **Getting Requirements**

This uses Python 3.10.x 

In [13]:
! pip install matplotlib pandas mplfinance marketprofile plotly yfinance-cache
! pip install yfinance --upgrade --no-cache-dir
! pip install --upgrade nbformat
! pip install IPython IPython notebook tabulate textblob


# **Imports**

In [14]:
# Standard library imports
from datetime import datetime, timedelta
import concurrent.futures
import logging
logging.basicConfig(level=40)

# Third-party imports
import json

import matplotlib.pyplot as plt
from tabulate import tabulate
from textblob import TextBlob

import mplfinance as mpf
import numpy as np
import time
from tqdm import tqdm
import nbformat
print(nbformat.__version__)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown
from IPython.display import display
import plotly.graph_objects as go

import yfinance as yf
# import yfinance_cache as yfc

# Local application/library specific imports
from market_profile import MarketProfile

5.10.3


# **Get all Companies from CSV and Fetch Initial Data**

In [15]:
file_path = './tickers.csv'  # Updated to a relative path for broader applicability

try:
    # Try loading the file. Adjust encoding if needed.
    data = pd.read_csv(file_path, encoding='ISO-8859-1')

    # Define the set of expected columns for validation.
    expected_columns = {'ticker'}

    # Check if the expected columns are present in the loaded data.
    if set(data.columns) & expected_columns == expected_columns:
        print('CSV file loaded successfully with the expected headers.')
        companies = data['ticker'].tolist()
    else:
        missing_columns = expected_columns - set(data.columns)
        print(f'CSV file is missing the following required columns: {missing_columns}')
        companies = []
except Exception as e:
    print(f'An error occurred while loading the CSV file: {e}')
    companies = []

# print("Total number of initial companies is: " + len(companies))

# Adjust or confirm the cache folder path as per your environment.
cache_folder = './colab_cache/'  # Adjusted to a more generic path unless you are specifically working within Google Colab.

def populate_metrics(ticker, metrics):
    if ticker and hasattr(ticker, 'info'):
        stock_info = ticker.info
        metrics['eps_values'].append(stock_info.get('trailingEps', 0))
        metrics['pe_values'].append(stock_info.get('trailingPE', 0))
        metrics['peg_values'].append(stock_info.get('trailingPegRatio', 0))
        metrics['gross_margins'].append(stock_info.get('grossMargins', 0))
        metrics['company_labels'].append(ticker.ticker)
    else:
        print(f"Skipped a company ticker due to missing info or an invalid object.")

def worker(company, metrics):
    try:
        ticker = yf.Ticker(company)
        populate_metrics(ticker, metrics)
    except Exception as e:
        print(f"Failed to fetch data for {company}. Error: {e}")

def fetch_metrics_data(companies):
    metrics = {metric: [] for metric in ['eps_values', 'pe_values', 'peg_values', 'gross_margins', 'company_labels']}

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(worker, company, metrics) for company in companies]

        for i, future in enumerate(tqdm(concurrent.futures.as_completed(futures), total=len(companies), desc="Fetching metrics")):
            # This loop is primarily to keep tqdm updated, handling of results (if any) would go here

            # Pause for 2 seconds after every 500 requests
            if i != 0 and i % 1000 == 0:
                time.sleep(30)

    return metrics

# Example of usage
metrics = fetch_metrics_data(companies)

CSV file loaded successfully with the expected headers.


Fetching metrics: 100%|██████████| 3350/3350 [00:36<00:00, 92.16it/s] 


# **Parametters**

In [16]:
days_history = 365 * 5  
interval_dates = '3mo'

eps_threshold = 5
gross_margin_threshold = 0.5 # 0.5 means 50%

peg_threshold_low = -0.1
peg_threshold_high = 1.1

def get_date_range(days_back):
    """Helper function to compute start and end date strings."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)
    return start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')

start_date_str, end_date_str = get_date_range(days_history)

# **Filtering Methods**

In [17]:
def filter_companies(metrics, eps_threshold, peg_threshold_low, peg_threshold_high, gross_margin_threshold):
    try:
        # Create DataFrame from metrics
        df = pd.DataFrame({
            'company': metrics['company_labels'],
            'eps': metrics['eps_values'],
            'pe': metrics['pe_values'],
            # 'ps': metrics['ps_values'],
            # 'pb': metrics['pb_values'],
            'peg': metrics['peg_values'],
            'gross_margin': metrics['gross_margins']
        })

        # Normalize gross margin if max is less than or equal to 1 (assuming it's in decimals rather than percentage)
        if df['gross_margin'].max() <= 1:
            df['gross_margin'] *= 100

        # Criteria for filtering - each condition needs to be enclosed in parentheses
        criteria = (df['eps'] > eps_threshold) & \
                   (df['gross_margin'] > gross_margin_threshold) & \
                   (df['peg'] > peg_threshold_low) & (df['peg'] <= peg_threshold_high) 

        # Apply criteria to filter DataFrame
        filtered_df = df[criteria]

        # Sort filtered DataFrame by 'pe' in ascending order
        filtered_df_sorted = filtered_df.sort_values(by='pe', ascending=True)

        # Print the filtered DataFrame
        print(f"Filtered down to {len(filtered_df_sorted)} companies based on criteria.")

        return filtered_df_sorted
    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()

def classify_by_industry(tickers):
    industries = {}
    for ticker in tickers.tickers.values():
        sector = ticker.info.get('sector')
        if sector:
            industries.setdefault(sector, []).append(ticker.ticker)
    return industries

def fetch_industries(companies):
    tickers = yf.Tickers(' '.join(companies))
    industries = classify_by_industry(tickers)
    return industries

def fetch_recommendations_summary(ticker):
    """Fetch a structured summary of recommendations for a given ticker."""
    try:
        rec_data = ticker.get_recommendations_summary()
        if not rec_data.empty:
            return {row['period']: {
                'strongBuy': row['strongBuy'],
                'buy': row['buy'],
                'hold': row['hold'],
                'sell': row['sell'],
                'strongSell': row['strongSell']
            } for index, row in rec_data.iterrows()}
        else:
            return "No recommendation data available."
    except Exception as e:
        return f"Error: {str(e)}"


def populate_additional_metrics(ticker, metrics):
    """Populates a given metrics dictionary with data fetched from a ticker object."""
    stock_info = ticker.info

    # Existing metrics enhancements
    metrics['recommendations_summary'].append(fetch_recommendations_summary(ticker))  
    metrics['ps_values'].append(stock_info.get('priceToSalesTrailing12Months', 0))
    metrics['pb_values'].append(stock_info.get('priceToBook', 0))
    metrics['market_caps'].append(stock_info.get('marketCap', 0))

    # Additional metrics as per the new requirements
    fields_to_add = {
        'forwardPE': 'forwardPE',
        'profitMargins': 'profitMargins',
        'heldPercentInsiders': 'heldPercentInsiders',
        'heldPercentInstitutions': 'heldPercentInstitutions',
        'forwardEps': 'forwardEps',
        'recommendationMean': 'recommendationMean',
        'recommendationKey': 'recommendationKey',
        'numberOfAnalystOpinions': 'numberOfAnalystOpinions',
        'totalCashPerShare': 'totalCashPerShare',
        'debtToEquity': 'debtToEquity',
        'freeCashflow': None,  # Will be fetched from df
        'earningsGrowth': 'earningsGrowth',
        'revenueGrowth': 'revenueGrowth'
    }

    df = ticker.cashflow
    
    # We are assuming df has a row corresponding to the specific ticker we are fetching for
    free_cash_flow = df.iloc[0, :].tolist()  # Change to df.iloc[:, 0].values for np.array

    for key, value in fields_to_add.items():
        if key not in metrics:
            metrics[key] = []  # Initialize if key doesn't exist
        if key == 'freeCashflow':
            metrics[key].append(free_cash_flow)
        else:
            metrics[key].append(stock_info.get(value, 0))  # Append with a default of 0 if not found

    return metrics

def augment_metrics_with_live_data(companies, original_metrics):
    augmented_data = {metric: [] for metric in original_metrics}  # Prepare structure
    augmented_data.update({  # Prepare for additional data
        'recommendations_summary': [],
        'news': [],
        'ps_values': [],
        'pb_values': [],
        'market_caps': []
    })

    for company_symbol in companies:
        try:
            # Assuming you have a way to get a ticker object for the company symbol
            ticker = get_ticker_object(company_symbol)  # You'll need to define this based on your data source/API

            # Initialize an empty dict for individual company metrics
            company_metrics = {metric: [] for metric in augmented_data}

            # Use populate_additional_metrics to fill company_metrics with live data
            populate_additional_metrics(ticker, company_metrics)

            # Now, append or extend each key in company_metrics to augmented_data
            for key, value in company_metrics.items():
                if isinstance(value, list) and key in augmented_data:  # Handling list types specifically
                    augmented_data[key].extend(value)
                else:
                    augmented_data[key].append(value)
        except Exception as e:
            print(f"Error processing company {company_symbol}: {e}")
            # Handle missing data or errors as needed, possibly skipping or using placeholder data

    return augmented_data

def get_ticker_object(symbol):
    """
    Given a symbol, returns a Ticker object using the yfinance library.
    :param symbol: The stock symbol (e.g., 'AAPL' for Apple Inc.)
    :return: yfinance.Ticker object containing information about the symbol
    """
    ticker = yf.Ticker(symbol)
    return ticker

def fetch_additional_metrics_data(companies):
    """Fetches and structures various financial metrics for the given list of company tickers."""
    tickers = yf.Tickers(' '.join(companies))
    metrics = {metric: [] for metric in ['ps_values', 'pb_values',
                                     'market_caps', 'recommendations_summary']}
    metrics['price_diff'] = {}

    for company in companies:
        try:
            ticker = tickers.tickers[company]
            populate_additional_metrics(ticker, metrics)
        except KeyError:
            print(f"Warning: Ticker {company} not found. Skipping.")

    return metrics

def build_combined_metrics(filtered_company_symbols, metrics, metrics_filtered):
    # Remove 'companies_fetched' if present in both dictionaries
    metrics.pop('companies_fetched', None)
    metrics_filtered.pop('companies_fetched', None)

    # Initialize combined_metrics excluding 'company_labels' explicitly as it's handled separately
    combined_metrics = {key: [] for key in set(list(metrics.keys()) + list(metrics_filtered.keys())) - {'company_labels', 'companies_fetched'}}

    # Directly use 'filtered_company_symbols' as the authoritative list of companies
    combined_metrics['company_labels'] = filtered_company_symbols

    for symbol in filtered_company_symbols:
        # Derive index from 'company_labels' in 'metrics' if it exists
        metrics_index = metrics['company_labels'].index(symbol) if 'company_labels' in metrics and symbol in metrics['company_labels'] else -1

        for key in combined_metrics:
            if key == 'company_labels':  # Skip 'company_labels' here since it's already handled
                continue

            # Process metrics from the 'metrics' dictionary
            if key in metrics and metrics_index >= 0:
                if isinstance(metrics[key], list) and len(metrics[key]) > metrics_index:
                    combined_metrics[key].append(metrics[key][metrics_index])
                else:
                    combined_metrics[key].append(None)  # Handle missing or misaligned data

            # Process additional metrics from the 'metrics_filtered' dictionary
            elif key in metrics_filtered:
                # We assume 'metrics_filtered' directly mirrors 'filtered_company_symbols' in sequence
                filtered_index = filtered_company_symbols.index(symbol)
                if isinstance(metrics_filtered[key], list) and len(metrics_filtered[key]) > filtered_index:
                    combined_metrics[key].append(metrics_filtered[key][filtered_index])
                else:
                    combined_metrics[key].append(None)  # Handle missing or misaligned data
            else:
                # If the key doesn't exist in 'metrics' when processing, initialize missing entries with None
                if key not in metrics:
                    combined_metrics[key].append(None)

    return combined_metrics

filtered_companies_df = filter_companies(metrics, eps_threshold, peg_threshold_low, peg_threshold_high, gross_margin_threshold)

filtered_company_symbols = filtered_companies_df['company'].tolist()

metrics_filtered = fetch_additional_metrics_data(filtered_company_symbols)

combined_metrics = build_combined_metrics(filtered_company_symbols, metrics, metrics_filtered)

filtered_industries = fetch_industries(filtered_company_symbols)

Filtered down to 12 companies based on criteria.


In [18]:
def fetch_historical_data(ticker, start_date, end_date, period=None, interval=interval_dates):
    try:
        if period:
            data = ticker.history(period=period, interval=interval)
        else:
            data = ticker.history(start=start_date, end=end_date, interval=interval)
        return data
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame

def calculate_price_diff(companies):
    tickers = yf.Tickers(' '.join(companies))
    price_diff = {}  # Store price difference info here

    for company in companies:
        ticker = tickers.tickers[company]
        hist = fetch_historical_data(ticker, None, None, period="1y")
        if not hist.empty:
            today_price = hist['Close'].iloc[-1]
            high_52week = max(hist['High'])
            low_52week = min(hist['Low'])
            high_percent_diff = ((today_price - high_52week) / high_52week) * 100
            low_percent_diff = ((today_price - low_52week) / low_52week) * 100
            price_diff[company] = {'high_diff': -1 * high_percent_diff, 'low_diff': low_percent_diff}

    return price_diff

def fetch_price_diff(companies, combined_metrics):
    price_diff = calculate_price_diff(companies)
    combined_metrics['price_diff'] = price_diff

    return combined_metrics

combined_metrics = fetch_price_diff(filtered_company_symbols, combined_metrics)

def plot_combined_interactive(combined_metrics):
    # Extract data for all plots
    company_labels = combined_metrics['company_labels']
    eps_values = combined_metrics['eps_values']
    high_diffs = [combined_metrics['price_diff'][company]['high_diff'] for company in company_labels]
    low_diffs = [combined_metrics['price_diff'][company]['low_diff'] for company in company_labels]
    market_caps = combined_metrics['market_caps']
    pb_values = combined_metrics['pb_values']
    pe_values = combined_metrics['pe_values']
    peg_values = combined_metrics['peg_values']
    ps_values = combined_metrics['ps_values']
    gross_margins = combined_metrics['gross_margins']
    recommendations_summary = combined_metrics['recommendations_summary']
    earningsGrowth = combined_metrics['earningsGrowth']
    revenueGrowth = combined_metrics['revenueGrowth']
    freeCashflow = combined_metrics['freeCashflow']

    # Normalize PEG sizes for Plot 4 visualization
    peg_min = min(peg_values)
    peg_max = max(peg_values)
    norm_peg_sizes = [(peg - peg_min) / (peg_max - peg_min) * 30 + 10 for peg in peg_values]

    fig = make_subplots(rows=3,  # Updated to 4 rows to include the desired 4th row
                            cols=3,
                            subplot_titles=("Price Difference % Over the Last Year",
                                            "EPS vs P/E Ratio",
                                            "Gross Margin (%)",
                                            "EPS vs P/B Ratio",
                                            "EPS vs PEG Ratio",
                                            "EPS vs P/S Ratio",
                                            "Upgrades & Downgrades Timeline",
                                            "Free Cash Flow",
                                            "Earnings Growth vs Revenue Growth",
                                            "Large Plot Spanning All Columns in 4th Row"),  # Added title for the large plot in the 4th row
                            specs=[
                                [{}, {}, {}],  # First row: 3 individual cells for plots
                                [{}, {}, {}],  # Second row: 3 individual cells for plots
                                [{}, {}, {}]  # Third row: 3 individual cells for plots
                            ],
                            vertical_spacing=0.1  # Adjust spacing for aesthetics
                        )

    colors = {company: f'hsl({(i / len(company_labels) * 360)},100%,50%)' for i, company in enumerate(company_labels)}

    ## sorting the gross margin bar charts
    sorted_data = sorted(zip(company_labels, gross_margins, colors), key=lambda x: x[1], reverse=True)  # Sort by gross_margin in descending order
    sorted_labels, sorted_gross_margins, sorted_colors = zip(*sorted_data)

    for i, company in enumerate(company_labels):
        legendgroup = f"group_{company}"
        marker_size = max(market_caps[i] / max(market_caps) * 50, 5)

        # Plot 1: Price Difference
        fig.add_trace(
            go.Scatter(
                x=[high_diffs[i]],
                y=[low_diffs[i]],
                marker=dict(size=10, color=colors[company]),
                legendgroup=legendgroup,
                name=company,
                hoverinfo='none',  # Disables default hover info, to use hovertemplate completely.
                hovertemplate=f'Company: {company}<br>High Diff: %{{x}}<br>Low Diff: %{{y}}<extra></extra>',  # Custom hover
            ),
            row=1, col=1
        )

        # Plot 2: EPS vs P/E Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[pe_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/E Ratio: {pe_values[i]}<extra></extra>',  # Custom hover
            ),
            row=1, col=2
        )

        # Plot 3: Gross Margin Bar Chart
        fig.add_trace(go.Bar(
            x=[company_labels[i]],
            y=[gross_margins[i] * 100],
            marker=dict(color=colors[company]),
            legendgroup=legendgroup,
            showlegend=False,
            width=0.8  # Adjust this value as needed
        ), row=1, col=3)

        # Plot 4:  EPS vs P/B Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[pb_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/B Ratio: {pb_values[i]}<extra></extra>',  # Custom hover
            ),
            row=2, col=1
        )

        # Plot 5:  EPS vs PEG Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[peg_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>PEG Ratio: {peg_values[i]}<extra></extra>',
            ),
            row=2, col=2
        )

        # Plot 6:  EPS vs P/S Ratio
        fig.add_trace(
            go.Scatter(
                x=[eps_values[i]],
                y=[ps_values[i]],
                marker=dict(size=marker_size, color=colors[company]),
                legendgroup=legendgroup,
                showlegend=False,
                hoverinfo='none',
                hovertemplate=f'Company: {company}<br>EPS: ${eps_values[i]}<br>P/S Ratio: {ps_values[i]}<extra></extra>',  # Custom hover
            ),
            row=2, col=3
        )

        # Plot 7: Recommendations Summary
        current_recommendations = recommendations_summary[i]

        if isinstance(current_recommendations, dict) and '0m' in current_recommendations:
            ratings = current_recommendations['0m']
            rating_categories = ['strongBuy', 'buy', 'hold', 'sell', 'strongSell']
            rating_values = [ratings.get(category, 0) for category in rating_categories]

            # Preparing data for bar plot
            y_categories = list(rating_categories) # Convert category names to numeric values
            category_names = {index: name for index, name in enumerate(rating_categories)} # Map for axis ticks

            # Use the count as bar height
            bar_heights = rating_values

            # Add bar chart to the subplot
            fig.add_trace(
                go.Bar(
                    x=rating_categories,  # Categories on the x-axis
                    y=bar_heights,  # Corresponding values on the y-axis
                    marker=dict(color=colors[company]),
                    name=company,
                    legendgroup=legendgroup,
                    showlegend=False,
                    text=company,  # Display company names
                    hoverinfo='y+text'  # Show hover text and value
                ),
                row=3, col=1
            )

            fig.update_yaxes(range=[0, max(peg_values)], row=2, col=2)

            # If you want to apply a starting y-axis of 0 universally to all subplots, you could do:
            for row in range(1, 3):  # Adjust the range accordingly based on your actual number of rows
                for col in range(1, 3):  # Adjust the range accordingly based on your actual number of columns
                    # Use conditions or specific logic if some charts shouldn't start from 0
                    fig.update_yaxes(range=[0, "auto"], row=row, col=col)

        else:
            # Handle unexpected data structure
            continue

        # # Adjust the years to match the length of the data
        now = datetime.now()
        year = now.year

        # If today's date is before April first, we are still in Q1
        if now.month < 4:
            year -= 1

        years = [str(year - i) for i in range(3, -1, -1)]
        
        fig.add_trace(            
            go.Scatter(                
                x=years[:len(freeCashflow[i])],                 
                y=[cf for cf in freeCashflow[i]],                
                # mode="lines",                
                name=company_labels[i],  # Legend title                
                hoverinfo='none',                
                legendgroup=legendgroup,     
                showlegend=False,
                hovertemplate=f'Company: {company}<br>Year: %{{x}}<br> Free Cashflow: %{{y}}<extra></extra>',            
            ),            
            row=3, col=2        
        )         

        fig.add_trace(            
            go.Scatter(                
                x=[revenueGrowth[i]],                
                y=[earningsGrowth[i]],                
                marker=dict(size=10, color=colors[company]),                
                legendgroup=legendgroup,                
                showlegend=False,  # Hide from legend                
                hoverinfo='none',                
                hovertemplate=f'Company: {company}<br>Revenue Growth: {revenueGrowth[i]}<br>Earnings Growth: {earningsGrowth[i]}<extra></extra>',
            ),            
            row=3, col=3        
        )

    # Update axes titles
    titles = [
        ("High Diff (%)", "Low Diff (%)"),  # Plot 1
        ("EPS", "P/E Ratio"),  # Plot 2
        ("Company", "Gross Margin (%)"),  # Plot 3
        ("Price to Books", "EPS"),  # Plot 4
        ("PEG", "EPS"),  # Plot 5
        ("P/S", "EPS"),  # Plot 6
        ("Years", "Free Cash Flow"),  # Plot 7 (3,2) - assumed Free Cash Flow plot
        ("Earnings Growth", "Revenue Growth")  # Plot 8 (3,3)
        # Assuming Plot 9 doesn't require a title because of the 'colspan' attribute
    ]


    for col, (x_title, y_title) in enumerate(titles, start=1):
        fig.update_xaxes(title_text=x_title, row=1, col=col)
        fig.update_yaxes(title_text=y_title, row=1, col=col)

    fig.update_xaxes(title_text="Recommendation Type", row=1, col=4)
    fig.update_yaxes(title_text="Number of Recommendations", row=1, col=4)

    fig.update_layout(height=1500)

    # Layout adjustments for readability and aesthetics
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=list([
                    dict(
                        args=[{"visible": "legendonly"}],  # This sets non-selected traces to be hidden.
                        label="Hide All",
                        method="restyle"
                    ),
                    dict(
                        args=[{"visible": True}],  # This makes all traces visible.
                        label="Show All",
                        method="restyle"
                    ),
                ]),
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0,
                xanchor="left",
                y=-0.15,
                yanchor="top"
            ),
        ]
    )
    # Show the combined plot
    fig.show()
    
# Extract company labels from 'price_diff' keys or another consistent source
company_labels = list(combined_metrics['company_labels'])

# Explicitly add 'company_labels' to 'metrics_filtered'
combined_metrics['company_labels'] = company_labels

plot_combined_interactive(combined_metrics)

# **Plot Sectors**

In [19]:
def plot_sector_distribution_interactive(industries, title):
    sector_counts = {sector: len(tickers) for sector, tickers in industries.items()}

    labels = list(sector_counts.keys())
    sizes = list(sector_counts.values())

    fig = go.Figure(data=[go.Pie(labels=labels, values=sizes, hole=.3)])

    fig.update_layout(
        title_text=title,
        annotations=[dict(text='Sectors', x=0.50, y=0.5, font_size=20, showarrow=False)]
    )

    fig.show()

plot_sector_distribution_interactive(filtered_industries, "Interactive Ticker Distribution by Sector for Filtered Tickers")
# plot_sector_distribution_interactive(industries, "Interactive Ticker Distribution by Sector for all Tickers")


# **Plotting charts**


In [22]:
print(combined_metrics)

def get_eps_pe_pb_ps_peg(ticker_symbol):
    try:
        if ticker_symbol in combined_metrics['company_labels']:
            index = combined_metrics['company_labels'].index(ticker_symbol)
            eps = combined_metrics['eps_values'][index]
            pe = combined_metrics['pe_values'][index]
            ps = combined_metrics['ps_values'][index]
            pb = combined_metrics['pb_values'][index]
            peg = combined_metrics['peg_values'][index]

            return eps, pe, ps, pb, peg
        else:
            print(f"Ticker '{ticker_symbol}' not found in the labels list.")
            return None, None, None, None, None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None, None, None, None

def calculate_market_profile(data):
    mp = MarketProfile(data)
    mp_slice = mp[data.index.min():data.index.max()]

    va_high, va_low = mp_slice.value_area
    poc_price = mp_slice.poc_price
    profile_range = mp_slice.profile_range

    return va_high, va_low, poc_price, profile_range

def plot_with_volume_profile(ticker_symbol, start_date, end_date):
    ticker = yf.Ticker(ticker_symbol)
    data = fetch_historical_data(ticker, start_date, end_date)

    eps, pe, ps, pb, peg = get_eps_pe_pb_ps_peg(ticker_symbol)

    if not data.empty:
        va_high, va_low, poc_price, _ = calculate_market_profile(data)

        poc_line = pd.Series(poc_price, index=data.index)
        va_high_line = pd.Series(va_high, index=data.index)
        va_low_line = pd.Series(va_low, index=data.index)

        apds = [mpf.make_addplot(poc_line, type='line', color='red', linestyle='dashed', width=3),
                mpf.make_addplot(va_high_line, type='line', color='blue', linestyle='dashed', width=0.7),
                mpf.make_addplot(va_low_line, type='line', color='blue', linestyle='dashed', width=0.7)]

        title = (f"{ticker.info['shortName']}\n\n"
                 f" EPS={eps}, P/E={pe}, P/S={ps}, \n P/B={pb}, PEG ratio={peg}\n\n\n")

        mpf.plot(data, type="candle", addplot=apds, title=title, style="yahoo", volume=True, show_nontrading=False)
    else:
        print(f"No data found for {ticker_symbol} in the given date range.")

def plot_candle_charts_per_sector(industries, start_date, end_date):
    for sector, symbol_list in industries.items():
        display(Markdown(f"# Sector: {sector}"))
        for symbol in symbol_list:
            display(Markdown(f"## {symbol} - {yf.Ticker(symbol).info['shortName']}"))
            
            # Website
            print(yf.Ticker(symbol).info['website'])
            
            # Market Cap
            display(Markdown(f"Market Cap: {yf.Ticker(symbol).info['marketCap']/1000000} Millions USD"))

            # News
            ndata = yf.Ticker(symbol).news
            news_data = []
            for item in ndata:
                publish_datetime = datetime.fromtimestamp(item['providerPublishTime'])
                now = datetime.now()
                days_ago = (now - publish_datetime).days
                blob = TextBlob(item['title'])
                polarity = blob.sentiment.polarity
                news_data.append({
                    'Title': f"[{item['title']}]({item['link']})",
                    'Publisher': item['publisher'], 
                    'Sentiment': polarity,
                    'Days Ago': days_ago
                })

            table_str = tabulate(news_data, headers="keys", tablefmt="pipe", showindex="always")
            display(Markdown(table_str))

            # Calendar
            data = yf.Ticker(symbol).calendar
            data['Earnings Date'] = [date.strftime('%B %d, %Y') for date in data['Earnings Date']]
            for key, value in data.items():
                if key.startswith('Revenue') and isinstance(value, int):
                    data[key] = '{:,.2f}'.format(value)
            table = [[key] + [val] for key, val in data.items()]
            table_str = tabulate(table, headers=["Metric", "Value"], tablefmt="pipe")
            display(Markdown(table_str))

            # Business Desc
            display(Markdown(f"{yf.Ticker(symbol).info['longBusinessSummary']}"))
            
            # Plot Chart
            plot_with_volume_profile(symbol, start_date, end_date)
            
            plt.tight_layout()
    
    plt.show()

plot_candle_charts_per_sector(filtered_industries, start_date_str, end_date_str)

{'earningsGrowth': [1.379, 1.36, 0.013, -0.716, 0.318, 0, -0.373, 0.41, -0.165, 0.106, 0.181, -0.803], 'forwardEps': [5.86, 7.73, 6.75, 24.06, 4.59, 12.05, 17.27, 11.12, 15.48, 11.02, 16.89, 204.11], 'heldPercentInstitutions': [1.0162001, 1.04833, 0.97602, 0.98314005, 1.02391, 1.02734, 0.98876, 0.39859, 0.97016996, 0.71283996, 0.95781, 0.97129995], 'recommendationMean': [2.0, 2.0, 2.4, 1.8, 1.2, 2.1, 1.9, 1.9, 2.5, 2.4, 2.2, 2.2], 'market_caps': [3937073152, 3648495616, 5636878848, 5369377792, 2673628416, 8012298240, 5507679232, 190623186944, 18649649152, 374070149120, 107368300544, 123860647936], 'profitMargins': [0.52835, 0.20382999, 0.36049, 0.32700002, 0.38951, 0.05756, 0.07014, 0.10587, 0.062080003, 0.41279, 0.36676, 0.20075001], 'gross_margins': [0.80136, 0.54825, 0.81442, 0.559, 0.86925006, 0.56990004, 0.56923, 0.62567, 0.87748003, 0.69181997, 0.55162996, 0.84582], 'price_diff': {'NOG': {'high_diff': 10.5637045710983, 'low_diff': 45.0390152382343}, 'CRC': {'high_diff': 9.2060187

# Sector: Energy

## NOG - Northern Oil and Gas, Inc.

https://www.noginc.com


Market Cap: 3937.073152 Millions USD

JSONDecodeError: Expecting value: line 1 column 1 (char 0)